# 09 — Rescue/harm transitions, prespecified ensemble, and manuscript exports

Case-level rescue/harm is measured against single-pass RAG. The exploratory ensemble uses the prespecified MedGemma-4B, Qwen2-1.5B, and Med-Qwen2-7B per-label majority vote. It has label metrics only—no synthetic report, text score, or RadGraph score.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
import importlib, json
import numpy as np, pandas as pd
from rerun_code.common import read_jsonl, write_jsonl
from rerun_code.config import sha256_path
from rerun_code.metrics import majority_vote, aggregate_label_metrics
import rerun_code.statistics as statistics_module
statistics_module = importlib.reload(statistics_module)
if int(getattr(statistics_module, "STATISTICS_API_VERSION", 0)) < 3:
    raise ImportError(
        "Notebook 09 requires the vectorized statistics engine (API 3). "
        "Copy rerun_code/statistics.py, restart the kernel, and rerun this cell."
    )
from rerun_code.statistics import paired_cluster_permutation, holm_adjust

# Do not export any downstream analysis from stale or incomplete
# upstream results. Notebook 08 must have analyzed the exact cohort
# produced by Notebook 07, including the direct F1-RadGraph outcome.
notebook07_status_path = PATHS["metrics"] / "notebook07_status.json"
notebook08_status_path = PATHS["statistics"] / "notebook08_status.json"
if not notebook07_status_path.exists() or not notebook08_status_path.exists():
    raise FileNotFoundError(
        "Run Notebooks 07 and 08 before Notebook 09; their status files are required."
    )
notebook07 = json.loads(notebook07_status_path.read_text(encoding="utf-8"))
notebook08 = json.loads(notebook08_status_path.read_text(encoding="utf-8"))
if not notebook07.get("ready") or not notebook08.get("ready"):
    raise RuntimeError(
        "Notebook 09 requires completed Notebooks 07 and 08. "
        f"Notebook 07 ready={notebook07.get('ready')}; "
        f"Notebook 08 ready={notebook08.get('ready')}."
    )
per_study_path = Path(notebook07.get("per_study_metrics", ""))
expected_sha256 = str(notebook07.get("per_study_metrics_sha256", ""))
if not per_study_path.exists() or not expected_sha256:
    raise FileNotFoundError("Notebook 07 per-study metrics or its checksum is missing.")
actual_sha256 = sha256_path(per_study_path)
if actual_sha256 != expected_sha256:
    raise AssertionError("Notebook 07 per-study metrics differ from notebook07_status.json.")
if notebook08.get("upstream_per_study_metrics_sha256") != actual_sha256:
    raise AssertionError(
        "Notebook 08 statistics are stale relative to Notebook 07 metrics. "
        "Rerun Notebook 08 before Notebook 09."
    )
text_audit_path = Path(notebook07.get("text_metric_runtime_audit", ""))
if not text_audit_path.exists():
    raise FileNotFoundError("Notebook 07 text-metric audit is missing.")
text_audit = json.loads(text_audit_path.read_text(encoding="utf-8"))
if text_audit.get("errors", {}).get("radgraph_f1"):
    raise RuntimeError(
        "Notebook 07 RadGraph failed; resolve the recorded error before final exports: "
        + str(text_audit["errors"]["radgraph_f1"])
    )
if text_audit.get("radgraph_f1_reward_component") != "partial_entity_relation_RG_ER":
    raise AssertionError(
        "Notebook 07 did not record the prespecified partial entity/relation "
        "F1-RadGraph component (RG_ER)."
    )

frame = pd.DataFrame(read_jsonl(per_study_path))
if "radgraph_f1" not in frame or not frame["radgraph_f1"].notna().any():
    raise RuntimeError("Notebook 07 per-study output has no usable radgraph_f1 values.")
frame["label_error_count"] = [int((np.asarray(r) != np.asarray(p)).sum()) for r, p in zip(frame["reference_vector"], frame["prediction_vector"])]
baseline = frame[frame["condition"] == "A_single_pass"][["model_key", "bundle", "source_dataset", "query_record_id", "label_error_count"]].rename(columns={"label_error_count": "baseline_error_count"})
transitions = frame.merge(baseline, on=["model_key", "bundle", "source_dataset", "query_record_id"], validate="many_to_one")
transitions["transition"] = np.where(transitions["label_error_count"] < transitions["baseline_error_count"], "rescue", np.where(transitions["label_error_count"] > transitions["baseline_error_count"], "harm", "unchanged"))
transitions.groupby(["model_key", "bundle", "source_dataset", "condition", "transition"]).size().rename("n").reset_index().to_csv(PATHS["analysis"] / "error_transitions.csv", index=False)

In [ ]:
ensemble_models = CONFIG["ensemble_models"]
ensemble_records = []
source = frame[frame["model_key"].isin(ensemble_models)]
for keys, group in source.groupby(["bundle", "source_dataset", "condition", "query_record_id"]):
    if set(group["model_key"]) != set(ensemble_models): continue
    first = group.iloc[0]
    ensemble_records.append({"bundle": keys[0], "source_dataset": keys[1], "condition": keys[2], "query_record_id": keys[3], "patient_key": first["patient_key"], "reference_vector": first["reference_vector"], "prediction_vector": majority_vote(group.set_index("model_key").loc[ensemble_models, "prediction_vector"].tolist())})
ensemble = pd.DataFrame(ensemble_records)
write_jsonl(PATHS["analysis"] / "ensemble_per_study.jsonl", ensemble.to_dict("records"))
ensemble_rows = []
for keys, group in ensemble.groupby(["bundle", "source_dataset", "condition"]):
    ensemble_rows.append({"bundle": keys[0], "source_dataset": keys[1], "condition": keys[2], **aggregate_label_metrics(group)})
ensemble_aggregate = pd.DataFrame(ensemble_rows)
ensemble_aggregate.to_csv(PATHS["analysis"] / "ensemble_label_metrics.csv", index=False)
ensemble_tests = []
label_primary = [metric for metric in CONFIG["statistics"]["primary_metrics"] if metric != "radgraph_f1"]
for keys, ensemble_group in ensemble.groupby(["bundle", "source_dataset", "condition"]):
    candidate = frame[(frame["bundle"] == keys[0]) & (frame["source_dataset"] == keys[1]) & (frame["condition"] == keys[2]) & frame["model_key"].isin(ensemble_models)]
    point_estimates = {model: aggregate_label_metrics(group)["macro_f1"] for model, group in candidate.groupby("model_key")}
    best_single = max(point_estimates, key=point_estimates.get)
    for model_key, single_group in candidate.groupby("model_key"):
        comparison = paired_cluster_permutation(single_group, ensemble_group, id_column="query_record_id", metrics=label_primary, replicates=int(CONFIG["statistics"]["permutation_replicates"]), seed=CONFIG["statistics"]["seed"])
        comparison["single_model"] = model_key; comparison["is_best_single_by_macro_f1"] = model_key == best_single
        comparison["bundle"], comparison["source_dataset"], comparison["condition"] = keys
        ensemble_tests.append(comparison)
ensemble_tests = pd.concat(ensemble_tests, ignore_index=True)
ensemble_tests["p_holm_ensemble_family"] = ensemble_tests.groupby(["metric", "bundle", "source_dataset", "condition"], dropna=False)["p_value"].transform(lambda values: holm_adjust(values.tolist()))
ensemble_tests.to_csv(PATHS["analysis"] / "ensemble_paired_tests.csv", index=False)

In [ ]:
runtime = frame.groupby(["model_key", "bundle", "source_dataset", "condition"])["runtime_seconds"].agg(["count", "mean", "median", "std", "min", "max"]).reset_index()
runtime.to_csv(PATHS["analysis"] / "runtime_summary.csv", index=False)
required = [PATHS["metrics"] / "aggregate_metrics.csv", PATHS["statistics"] / "cluster_bootstrap_ci.csv", PATHS["statistics"] / "paired_cluster_permutation_tests.csv", PATHS["statistics"] / "between_model_paired_tests.csv", PATHS["analysis"] / "error_transitions.csv", PATHS["analysis"] / "ensemble_label_metrics.csv", PATHS["analysis"] / "ensemble_paired_tests.csv", PATHS["analysis"] / "runtime_summary.csv"]
missing = [str(path) for path in required if not path.exists()]
if missing: raise AssertionError("Missing final exports:\n" + "\n".join(missing))
print("RERUN ANALYSIS COMPLETE. Manuscript-ready source tables are in", PATHS["metrics"], PATHS["statistics"], "and", PATHS["analysis"])